In [1]:
# load modules
import geopandas as gpd
import pandas as pd
import fiona
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

import matplotlib.pyplot as plt

In [2]:
# load the file with the narratives from the crash report

narratives1 = pd.read_csv("Data/Franklin_DocumentNumber_Narrative_part1.csv")
narratives2 = pd.read_csv("Data/Franklin_DocumentNumber_Narrative_part2.csv")

In [3]:
# merge the files in one
narratives = pd.concat([narratives1, narratives2], ignore_index=True)

In [4]:
narratives

,DocumentNumber,County,Narrative
0,20202168971,Franklin County,Unit# 1 a pedestrian was crossing US-40 from s...
1,20202168999,Franklin County,Unit 1 was westbound on SR 317 towards Parsons...
2,20202172825,Franklin County,Unit 2 was traveling south on SR 104 south of ...
3,20202183697,Franklin County,Unit 1 was crossing US 40 outside of the cross...
4,20202206898,Franklin County,Unit 1 was driving south on SR 3 just north of...
...,...,...,...
115676,20258172808,Franklin County,On Sunday; September 21; 2025; Unit 2 was stop...
115677,20258173076,Franklin County,On above date and time U-1 failed to yield at ...
115678,20258173087,Franklin County,Unit #1 was traveling eastbound on Josephus Ln...
115679,20258173091,Franklin County,During the night of September 13th; 2025 after...


In [4]:
# now search for the top 10 (or top 20) fast food chains in the narratives

# load the fastfood places
fastfood = gpd.read_file("Data/fastfood_all_ohio.gpkg", layer="fastfood")

# check the occurances of the pizza chains
fastfood_counts = fastfood["name"].value_counts()

# convert to a DataFrame for nicer display
fastfood_counts_df = fastfood_counts.reset_index()

print(fastfood_counts[:10])
top10_fastfood = fastfood_counts[:10]
top20_fastfood = fastfood_counts[:20]


# load the pizza places
pizza = gpd.read_file("Data/pizza_places_ohio.gpkg", layer="pizza_places")
# check the occurances of the pizza chains
pizza_counts = pizza["name"].value_counts()

# convert to dataframe for nicer display
pizza_counts_df = pizza_counts.reset_index()
#pizza_counts_df.columns = ["fastfood_chain", "count"]

print(pizza_counts[:10])
top10_pizza = pizza_counts[:10]
top20_pizza = pizza_counts[:20]

name
McDonald's     614
Subway         585
Wendy's        368
Taco Bell      325
Burger King    281
Arby's         245
Chipotle       214
Domino's       162
Dunkin'        162
KFC            156
Name: count, dtype: int64
name
Pizza Hut                174
Domino's                 161
Little Caesars            99
Papa John's               83
Marco's Pizza             83
Donatos Pizza             69
LaRosa's Pizzeria         30
Jet's Pizza               19
Gionino's Pizzeria        18
East of Chicago Pizza     18
Name: count, dtype: int64


In [5]:
pizza_10names = top10_pizza.index[:]
fastfood_10names = top10_fastfood.index[:]

print(pizza_10names)
print(fastfood_10names)

Index(['Pizza Hut', 'Domino's', 'Little Caesars', 'Papa John's',
       'Marco's Pizza', 'Donatos Pizza', 'LaRosa's Pizzeria', 'Jet's Pizza',
       'Gionino's Pizzeria', 'East of Chicago Pizza'],
      dtype='object', name='name')
Index(['McDonald's', 'Subway', 'Wendy's', 'Taco Bell', 'Burger King', 'Arby's',
       'Chipotle', 'Domino's', 'Dunkin'', 'KFC'],
      dtype='object', name='name')


In [6]:
# now we have the list of the top 10 restaurant names,
# now look through the narratives and search for these words
# issue might be that there is different writing of the restaurants in the narrative
# like Dominos instead of Domino's

# set up a new dictionary to store the information
narrative_names = {}

for i in range(len(fastfood_10names)):
    mask = narratives['Narrative'].str.contains(fastfood_10names[i],case=False,na=False)
    extracted_indices = narratives.index[mask]
    narrative_names[fastfood_10names[i]] = extracted_indices


In [7]:
narrative_names

{"McDonald's": Index([   570,   1678,   2661,   3368,   6295,   6653,   6695,   8329,  10437,
         11190,
        ...
        108245, 108811, 109209, 109315, 109747, 110592, 110922, 111859, 113591,
        113951],
       dtype='int64', length=111),
 'Subway': Index([  9162,  10623,  11334,  11351,  12494,  21815,  31826,  36111,  42586,
         44171,  45000,  54071,  54735,  60222,  60946,  72924,  86293,  92796,
         96996, 102597, 105448],
       dtype='int64'),
 "Wendy's": Index([  5872,   6552,   8333,   9675,   9677,  10079,  12411,  12505,  13115,
         13269,  14046,  16883,  17662,  17968,  17997,  18118,  18147,  20226,
         20965,  23107,  28000,  33703,  33905,  34626,  35009,  37652,  37866,
         39589,  42262,  43233,  46615,  47254,  51007,  55863,  55911,  56200,
         56207,  58824,  61743,  64831,  66434,  67112,  68636,  69311,  75162,
         80433,  82418,  84785,  93587,  94095,  94268,  95166,  96218,  96669,
         97029,  98623,  9882

In [8]:
print(len(narrative_names["McDonald\'s"]))
print(narrative_names["McDonald\'s"])


111
Index([   570,   1678,   2661,   3368,   6295,   6653,   6695,   8329,  10437,
        11190,
       ...
       108245, 108811, 109209, 109315, 109747, 110592, 110922, 111859, 113591,
       113951],
      dtype='int64', length=111)


In [9]:
print(narratives.iloc[108245])

Narrative    On 6/2/25 at approximately 14:45 pm; Officers ...
Name: 108245, dtype: object
